# AutoAlert: алертинг продуктовых метрик

Ход работы инструмента по шагам. Вся логика — в модулях [`src/`](src/), описание метода и формат входных данных — в [README](README.md).

In [ ]:
import re
from pathlib import Path

import pandas as pd

from src.data import prepare_metrics, iter_series
from src.detection import Params, backtest
from src.plots import plot_metrics, draw_spikes, draw_trend_alerts
from src.report import build_status_table, format_message

DATA_PATH = Path("data/metrics.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Тип метрики нужен, чтобы коридор нормы не уходил ниже нуля (счётчики) и за пределы [0, 1] (доли)
COUNT_METRICS = ["plan_count", "clients_with_open", "clients_with_add", "send_count", "deliver_count", "read_count"]
RATE_METRICS = ["conv_to_send", "conv_to_deliver", "conv_to_read", "coverage"]
METRIC_KINDS = {**dict.fromkeys(COUNT_METRICS, "count"), **dict.fromkeys(RATE_METRICS, "rate")}

params = Params()  # рекомендуемые значения, описание — в README
params

In [ ]:
def file_name(text):
    return re.sub(r"\W+", "_", text).strip("_")

## 1. Данные

Длинная таблица `date / metric / value / channel`. Каждая пара (канал, метрика) превращается в дневной ряд.

In [ ]:
metrics = prepare_metrics(pd.read_csv(DATA_PATH))
series = {(channel, metric): values for channel, metric, values in iter_series(metrics)}

print(f"Каналов: {metrics['channel'].nunique()}, рядов: {len(series)}")
print(f"Период: {metrics['date'].min():%Y-%m-%d} — {metrics['date'].max():%Y-%m-%d}")

## 2. Ежедневный запуск на истории

Имитируем работу по расписанию: для каждого дня детектор видит только данные до этого дня включительно и оценивает последнюю точку — есть ли выброс и сдвигается ли уровень. Так видно, какие алерты инструмент прислал бы на самом деле. Первые дни ряда не оцениваются: для сравнения с уровнем `trend_lookback_days` дней назад нужна история.

Детектор пересчитывается для каждого дня истории, поэтому шаг может занять несколько минут.

In [ ]:
history = pd.concat(
    [
        backtest(values, params, METRIC_KINDS.get(metric)).assign(channel=channel, metric=metric)
        for (channel, metric), values in series.items()
    ],
    ignore_index=True,
)

history.groupby(["channel", "metric"]).agg(
    spikes_up=("spike", lambda s: (s == "up").sum()),
    spikes_down=("spike", lambda s: (s == "down").sum()),
    trend_signals=("trend", lambda s: (s != "normal").sum()),
    trend_alerts=("trend_alert", "sum"),
)

## 3. Выбросы

Ожидаемый уровень и коридор нормы — такие, какими их видел детектор в каждый день, точки — выбросы, о которых он сообщил бы в этот день.

In [ ]:
for channel in metrics["channel"].unique():
    panels = dict(list(history[history["channel"] == channel].groupby("metric")))
    fig = plot_metrics(panels, draw_spikes, title=f"{channel}: выбросы при ежедневном запуске (LOWESS + MAD)")
    fig.savefig(OUTPUT_DIR / f"spikes_{file_name(channel)}.png", dpi=110)

## 4. Плавный сдвиг уровня

Бледные точки — дневные сигналы сдвига, кольца — алерты: сигнал продержался `confirm_days` дней подряд.

In [ ]:
for channel in metrics["channel"].unique():
    panels = dict(list(history[history["channel"] == channel].groupby("metric")))
    title = f"{channel}: алерты плавного сдвига ({params.confirm_days} дн. подряд)"
    fig = plot_metrics(panels, draw_trend_alerts, title=title)
    fig.savefig(OUTPUT_DIR / f"trend_alerts_{file_name(channel)}.png", dpi=110)

## 5. Сводка на последний день

Итог ежедневного запуска: статус каждой метрики и текст сообщения для рабочего чата.

In [ ]:
status = build_status_table(history)
status.to_csv(OUTPUT_DIR / "status.csv", index=False)
status

In [ ]:
message = format_message(status, lookback_days=params.trend_lookback_days)
print(message)

В рабочей версии сводка дописывается в историческую витрину, а сообщение отправляет бот в рабочий чат через внутреннюю библиотеку, поэтому здесь отправки нет.

Для ежедневного запуска не нужно прогонять всю историю — достаточно последних `confirm_days` дней:

```python
backtest(values, params, kind, start=len(values) - params.confirm_days)
```